# Notebook 02 — Data Loading and First Inspection

## Purpose
I load all 9 Olist CSV files and perform a first visual inspection of each.
This is the first time I see the actual data.

## Why this matters
I check shapes, dtypes, and .head() before any processing to catch immediate
problems (wrong separators, encoding issues, unexpected columns).

## Inputs
`data/raw/` — all 9 Olist CSV files

## Outputs
`data/interim/raw_loaded.parquet` — optional snapshot to avoid reloading CSVs

## Decisions I make here
- Confirm all 9 files loaded without errors
- Note any obvious issues (mixed types, wrong shapes, unexpected nulls)


In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

sys.path.insert(0, str(Path('..').resolve()))

from src.config import load_config
from src.paths import Paths
from src.utils import set_all_seeds

cfg   = load_config()
paths = Paths(cfg)
SEED  = cfg['project']['random_seed']
set_all_seeds(SEED)

print("Config and paths loaded.")
print(f"Raw data directory: {paths.raw}")


  Random seed set to 42
Config and paths loaded.
Raw data directory: C:\Users\Peter\Documents\projects\Jobberman_projects\double_Integral\ecommerce_customer_intelligence\data\raw


In [2]:
# I check that all raw files exist before loading
expected = cfg['data']['olist_files']
missing = []
for key, fname in expected.items():
    fp = paths.raw / fname
    if not fp.exists():
        missing.append(fname)

if missing:
    print("ERROR: Missing files. Run: python scripts/download_data.py")
    for f in missing:
        print(f"  MISSING: {f}")
else:
    print("All 9 expected files found in data/raw/. Proceeding to load.")


All 9 expected files found in data/raw/. Proceeding to load.


In [3]:
# Load all 9 CSV files
# I use parse_dates for timestamp columns to avoid doing it later
DATE_COLS_ORDERS = [
    'order_purchase_timestamp', 'order_approved_at',
    'order_delivered_carrier_date', 'order_delivered_customer_date',
    'order_estimated_delivery_date',
]

orders      = pd.read_csv(paths.raw / 'olist_orders_dataset.csv',
                          parse_dates=DATE_COLS_ORDERS)
items       = pd.read_csv(paths.raw / 'olist_order_items_dataset.csv')
payments    = pd.read_csv(paths.raw / 'olist_order_payments_dataset.csv')
reviews     = pd.read_csv(paths.raw / 'olist_order_reviews_dataset.csv')
customers   = pd.read_csv(paths.raw / 'olist_customers_dataset.csv')
products    = pd.read_csv(paths.raw / 'olist_products_dataset.csv')
sellers     = pd.read_csv(paths.raw / 'olist_sellers_dataset.csv')
geo         = pd.read_csv(paths.raw / 'olist_geolocation_dataset.csv')
translation = pd.read_csv(paths.raw / 'product_category_name_translation.csv')

tables = {
    'orders': orders, 'items': items, 'payments': payments,
    'reviews': reviews, 'customers': customers, 'products': products,
    'sellers': sellers, 'geolocation': geo, 'translation': translation,
}
print("Loaded tables:")
for name, df in tables.items():
    print(f"  {name:15s}  {df.shape[0]:>8,} rows  x  {df.shape[1]:>3} cols")


Loaded tables:
  orders             99,441 rows  x    8 cols
  items             112,650 rows  x    7 cols
  payments          103,886 rows  x    5 cols
  reviews            99,224 rows  x    7 cols
  customers          99,441 rows  x    5 cols
  products           32,951 rows  x    9 cols
  sellers             3,095 rows  x    4 cols
  geolocation      1,000,163 rows  x    5 cols
  translation            71 rows  x    2 cols


In [4]:
# First look: orders table
print("=== ORDERS ===")
print(orders.shape)
print()
display(orders.head(3))
print()
print(orders.dtypes)


=== ORDERS ===
(99441, 8)



,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04



order_id                                 object
customer_id                              object
order_status                             object
order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object


In [5]:
# First look: order_items
print("=== ORDER ITEMS ===")
print(items.shape)
display(items.head(3))
print(items.dtypes)


=== ORDER ITEMS ===
(112650, 7)


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.9,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.9,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.0,17.87


order_id                object
order_item_id            int64
product_id              object
seller_id               object
shipping_limit_date     object
price                  float64
freight_value          float64
dtype: object


In [6]:
# First look: reviews
print("=== REVIEWS ===")
print(reviews.shape)
display(reviews.head(3))
print()
print("Review score value counts:")
print(reviews['review_score'].value_counts().sort_index())


=== REVIEWS ===
(99224, 7)


,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24



Review score value counts:
review_score
1    11424
2     3151
3     8179
4    19142
5    57328
Name: count, dtype: int64


In [7]:
# First look: customers
print("=== CUSTOMERS ===")
print(customers.shape)
display(customers.head(3))
print()
# Important: customer_id is per-order, customer_unique_id is per-person
print("Unique customer_id:       ", customers['customer_id'].nunique())
print("Unique customer_unique_id:", customers['customer_unique_id'].nunique())
print()
print("This means some people placed more than one order (different customer_id, same unique_id)")


=== CUSTOMERS ===
(99441, 5)


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP



Unique customer_id:        99441
Unique customer_unique_id: 96096

This means some people placed more than one order (different customer_id, same unique_id)


In [8]:
# Quick scan of all tables
print("Summary scan:")
print(f"{'Table':<15} {'Rows':>8} {'Cols':>5} {'Nulls_total':>12}")
print("-" * 45)
for name, df in tables.items():
    nulls = df.isnull().sum().sum()
    print(f"{name:<15} {len(df):>8,} {df.shape[1]:>5} {nulls:>12,}")


Summary scan:
Table               Rows  Cols  Nulls_total
---------------------------------------------
orders            99,441     8        4,908
items            112,650     7            0
payments         103,886     5            0
reviews           99,224     7      145,903
customers         99,441     5            0
products          32,951     9        2,448
sellers            3,095     4            0
geolocation     1,000,163     5            0
translation           71     2            0


In [9]:
# Check date range for orders
print("Order date range:")
print(f"  Earliest purchase: {orders['order_purchase_timestamp'].min()}")
print(f"  Latest  purchase:  {orders['order_purchase_timestamp'].max()}")
print()
print("Order status breakdown:")
print(orders['order_status'].value_counts())


Order date range:
  Earliest purchase: 2016-09-04 21:15:19
  Latest  purchase:  2018-10-17 17:30:18

Order status breakdown:
order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64


In [10]:
# Save a quick snapshot so I can reload without parsing CSVs again
paths.interim.mkdir(parents=True, exist_ok=True)
orders.to_parquet(paths.interim / 'orders_raw.parquet', index=False)
print("Saved orders_raw.parquet to data/interim/")
print()
print("Notebook 02 complete.")
print("Next: Notebook 03 - Schema audit and column-by-column inspection")


Saved orders_raw.parquet to data/interim/

Notebook 02 complete.
Next: Notebook 03 - Schema audit and column-by-column inspection
